In [ ]:
!pip install transformers torch

In [ ]:
import json
def fix_notebook_metadata(file_path):
    with open(file_path, 'r') as f:
        data = json.load(f)
    if 'colab' in data['metadata']:
        data['metadata']['colab']['private_outputs'] = False
        print("Metadata updated: private_outputs is now False.")
    else:
        print("No Colab metadata found. You may need to save the notebook once first.")

    with open(file_path, 'w') as f:
        json.dump(data, f, indent=2)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
device = torch.device("cpu")
MODEL_ID = "google/flan-t5-base"

In [ ]:
def initialize_engine():
    """Load the instruction-tuned model and its tokenizer."""
    print(f"--- System: Loading {MODEL_ID} ---")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID).to(device)
    print(f"--- System: Engine Ready on {device} ---\n")
    return tokenizer, model

In [ ]:
def process_query(user_text, tokenizer, model):
    """
    Handles logic: Checks hardcoded facts first for accuracy,
    then falls back to the Transformer for general reasoning.
    """
    raw_query = user_text.lower().strip()
    knowledge_map = {
        "artificial intelligence": (
            "Artificial Intelligence (AI) involves creating systems capable of performing tasks "
            "that typically require human intelligence, such as reasoning, learning, and self-correction."
        ),
        "who created python": (
            "Python was conceived by Guido van Rossum at CWI in the Netherlands. "
            "It was first released in 1991 as a successor to the ABC language."
        )
    }

    for key, fact in knowledge_map.items():
        if key in raw_query:
            return fact
    instruction_prompt = f"Answer the following question as a professional assistant: {user_text}"

    inputs = tokenizer(instruction_prompt, return_tensors="pt").to(device)

    output_tokens = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        repetition_penalty=1.5
    )

    return tokenizer.decode(output_tokens[0], skip_special_tokens=True)

In [ ]:
def run_assistant():
    """Main loop for the interactive console-based chatbot."""
    tokenizer, model = initialize_engine()

    print("AI Assistant: Hello! I am your AI assistant. How can I assist you today?")
    print("(Type 'exit' or 'quit' to end session)\n")

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() in ["exit", "quit"]:
            print("AI Assistant: Goodbye! It was a pleasure helping you.")
            break

        if user_input.lower() in ["hi", "hello", "hey"]:
            print("AI Assistant: Hello! How can I help you today?")
            continue

        if "thank" in user_input.lower():
            print("AI Assistant: You're very welcome! Let me know if you need anything else.")
            continue

        bot_response = process_query(user_input, tokenizer, model)
        print(f"AI Assistant: {bot_response}")

if __name__ == "__main__":
    run_assistant()